# Lesson 5 — Multi-agent and integration

Many agents cooperating, plus a way to plug in tools we don't own.

Covers `s15` + `s16` + `s17` + `s19` + `s20`. Narration in `speaker_notes/05_multi_agent_and_integration.md`.

In [ ]:
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path

from llm_client import get_client, DEFAULT_MODEL

client = get_client()

## Part 1 — Message bus

JSONL inbox per agent. Send = append a line. Read = read + delete (destructive). Real Claude Code uses the same layout with `proper-lockfile`.

In [ ]:
MAILBOX_DIR = Path(".mailboxes_demo")
MAILBOX_DIR.mkdir(exist_ok=True)

class MessageBus:
    def send(self, from_agent: str, to_agent: str, content: str,
             msg_type: str = "message", metadata: dict | None = None):
        msg = {
            "from": from_agent, "to": to_agent, "content": content,
            "type": msg_type, "ts": time.time(),
            "metadata": metadata or {},
        }
        inbox = MAILBOX_DIR / f"{to_agent}.jsonl"
        with open(inbox, "a", encoding="utf-8") as f:
            f.write(json.dumps(msg) + "\n")

    def read_inbox(self, agent: str) -> list[dict]:
        inbox = MAILBOX_DIR / f"{agent}.jsonl"
        if not inbox.exists():
            return []
        msgs = [json.loads(line) for line in inbox.read_text().splitlines() if line.strip()]
        inbox.unlink()   # destructive: consume all messages atomically
        return msgs

BUS = MessageBus()

BUS.send("lead", "alice", "please investigate the failing test")
BUS.send("lead", "alice", "also check the deploy logs")
print("alice reads:", BUS.read_inbox("alice"))
print("alice reads again:", BUS.read_inbox("alice"))   # gone — destructive

## Part 2 — Protocols

Typed request/response handshakes. Each request has an ID and a type; the response must match both. Prevents an approval being matched to a shutdown.

In [ ]:
@dataclass
class ProtocolState:
    request_id: str
    type: str         # e.g. "plan_approval"
    sender: str
    target: str
    status: str = "pending"   # pending | approved | rejected
    payload: dict = field(default_factory=dict)

pending_requests: dict[str, ProtocolState] = {}

def send_plan_approval(from_agent: str, to_agent: str, plan: str) -> str:
    req_id = str(uuid.uuid4())[:8]
    pending_requests[req_id] = ProtocolState(
        request_id=req_id, type="plan_approval",
        sender=from_agent, target=to_agent, payload={"plan": plan},
    )
    BUS.send(from_agent, to_agent, plan,
             msg_type="plan_approval_request",
             metadata={"request_id": req_id})
    return req_id

def respond(responder: str, request_id: str, response_type: str, approve: bool):
    state = pending_requests.get(request_id)
    if not state:
        return f"no such request {request_id}"
    expected = f"{state.type}_response"
    if response_type != expected:
        return f"type mismatch: expected {expected}, got {response_type}"
    state.status = "approved" if approve else "rejected"
    BUS.send(responder, state.sender,
             "approved" if approve else "rejected",
             msg_type=response_type,
             metadata={"request_id": request_id, "approve": approve})
    return "ok"

In [ ]:
rid = send_plan_approval("lead", "alice", "refactor module X in 3 steps")
print("alice's inbox:", BUS.read_inbox("alice"))
respond("alice", rid, "plan_approval_response", approve=True)
print("lead's inbox:", BUS.read_inbox("lead"))
print("final state:", pending_requests[rid].status)

# Try to approve with the wrong type:
rid2 = send_plan_approval("lead", "alice", "do another thing")
BUS.read_inbox("alice")    # consume
print("wrong type:", respond("alice", rid2, "shutdown_response", approve=True))

## Part 3 — Autonomous agents

An agent that idles and polls for inbox messages or unclaimed tasks. Two wake-up sources, one loop.

In [ ]:
# Minimal task store inlined for this notebook (mirrors lesson 4's shape)
TASKS = {}

def create_task(description, owner=None):
    tid = str(uuid.uuid4())[:8]
    TASKS[tid] = {"id": tid, "description": description,
                  "status": "pending", "owner": owner}
    return tid

def unclaimed():
    return [t for t in TASKS.values() if t["status"] == "pending" and t["owner"] is None]

def claim(tid, owner):
    if TASKS[tid]["status"] != "pending" or TASKS[tid]["owner"]:
        return False
    TASKS[tid]["status"] = "in_progress"
    TASKS[tid]["owner"] = owner
    return True

def idle_poll(agent_name: str, *, poll_interval=1, timeout=10) -> dict:
    """Block until something happens or timeout. Returns a wake-up event."""
    elapsed = 0
    while elapsed < timeout:
        msgs = BUS.read_inbox(agent_name)
        if msgs:
            return {"reason": "inbox", "messages": msgs}
        tasks = unclaimed()
        if tasks:
            t = tasks[0]
            if claim(t["id"], agent_name):
                return {"reason": "claimed_task", "task": t}
        time.sleep(poll_interval)
        elapsed += poll_interval
    return {"reason": "timeout"}

In [ ]:
# Demo: start a poll, then a moment later create a task → poll wakes up
import threading

result = {}
def worker():
    result["event"] = idle_poll("alice", timeout=5)

threading.Thread(target=worker).start()
time.sleep(1.5)
create_task("investigate latency spike")
time.sleep(2)
print("alice woke up because:", result["event"])

## Part 4 — MCP plugins

External tool servers expose a `tools` list; we merge them into our pool with a `mcp__<server>__<tool>` prefix to avoid collisions. The model can't tell which is which.

In [ ]:
class MCPClient:
    def __init__(self, name: str, tools_with_handlers: dict):
        self.name = name
        self.tools = [{"name": n, "description": v["description"],
                       "parameters": v["parameters"]}
                      for n, v in tools_with_handlers.items()]
        self._handlers = {n: v["handler"] for n, v in tools_with_handlers.items()}

    def call_tool(self, tool_name: str, args: dict) -> str:
        handler = self._handlers.get(tool_name)
        if not handler:
            return f"unknown mcp tool: {tool_name}"
        return handler(**args)

# A pretend "docs" MCP server with one tool
docs_mcp = MCPClient("docs", {
    "search": {
        "description": "Search internal documentation.",
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string"}}, "required": ["query"]},
        "handler": lambda query: f"(fake docs result for: {query})",
    },
})

mcp_clients = {"docs": docs_mcp}

In [ ]:
# Builtin tools (just one for the demo)
BUILTIN_TOOLS = [
    {"type": "function", "function": {
        "name": "echo", "description": "Echo back a string.",
        "parameters": {"type": "object",
            "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
]
BUILTIN_HANDLERS = {"echo": lambda text: text}

def _safe_name(s: str) -> str:
    return s.replace("-", "_").replace(".", "_")

def assemble_tool_pool():
    tools = list(BUILTIN_TOOLS)
    handlers = dict(BUILTIN_HANDLERS)
    for server_name, mcp in mcp_clients.items():
        for td in mcp.tools:
            prefixed = f"mcp__{_safe_name(server_name)}__{td['name']}"
            tools.append({"type": "function", "function": {
                "name": prefixed, "description": td["description"],
                "parameters": td["parameters"]}})
            # Capture the right server + name in closure
            handlers[prefixed] = (
                lambda _mcp=mcp, _name=td["name"], **kw: _mcp.call_tool(_name, kw)
            )
    return tools, handlers

tools, handlers = assemble_tool_pool()
print("merged tool pool:")
for t in tools:
    print(" ", t["function"]["name"])

## Part 5 — Putting it all together  (`s20_comprehensive`)

The full Claude Code agent loop has the *same skeleton* you wrote in lesson 1, with each mechanism slotted in:

```text
while turn < max_turns:
    # lesson 3 — keep context bounded
    messages = micro_compact(snip_compact(messages))

    # lesson 4 — drain async signals INTO the conversation
    messages += collect_finished_notifications()
    messages += drain_cron_queue()

    # lesson 5 — drain inbox for multi-agent messages
    messages += [bus_msg_to_user(m) for m in BUS.read_inbox(self.name)]

    # lesson 2 — assemble system prompt from live state
    system = get_system_prompt(context)

    # lesson 3 — wrap the API call in error-recovery
    resp = call_with_recovery(messages, tools)

    # lesson 1 — dispatch tools, lesson 2 — through hooks
    for call in resp.tool_calls:
        result = run_tool_with_hooks(call, handlers)
        messages.append(tool_result(call.id, result))

    if finish_reason == 'stop':
        # lesson 5 — idle-poll for the next thing to do
        event = idle_poll(self.name)
        ...
```

Every layer is **orthogonal**: adding the next one doesn't change the previous ones. That is the whole point of the design — the harness is a stack of small, replaceable mechanisms around an unchanged core loop.

Let's wire up a tiny demo that combines: builtin tool + MCP tool + message bus.

In [ ]:
AGENT_NAME = "lead"

def run_one_turn(user_prompt: str):
    tools, handlers = assemble_tool_pool()
    messages = [
        {"role": "system", "content":
            f"You are agent '{AGENT_NAME}'. You can call builtins like 'echo' "
            "and MCP tools prefixed mcp__server__name (e.g. mcp__docs__search)."},
        {"role": "user", "content": user_prompt},
    ]
    for _ in range(4):
        # Drain inbox into the conversation
        for m in BUS.read_inbox(AGENT_NAME):
            messages.append({"role": "user", "content":
                f"<from {m['from']} type={m['type']}>\n{m['content']}"})

        resp = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages, tools=tools)
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if resp.choices[0].finish_reason != "tool_calls":
            return msg.content
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            handler = handlers.get(call.function.name)
            output = handler(**args) if handler else f"unknown: {call.function.name}"
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(output)})
    return "(max_turns)"

BUS.send("alice", AGENT_NAME, "FYI — I'm done with the test refactor.")
result = run_one_turn("Search the docs for 'observability', then echo a one-line summary of what alice told you.")
print("\nfinal:", result)

## Where to go next

You've built every mechanism Claude Code uses in production. The `s01`–`s20` files in this repo are the full version with edge cases — read them next.

**Suggested exercises**
1. Add `bash_background` (lesson 4) to the comprehensive loop here.
2. Convert lesson 1's permission check into a hook and apply it everywhere.
3. Implement `autoCompact` — an LLM call that summarizes the snipped middle.
4. Build a real MCP client that speaks JSON-RPC over stdio.